# Module 15: Multimodal LLMs

**Goal:** Handle more than just text — vision, image generation, audio, video.

**What you'll learn:**
- Vision APIs for image analysis and OCR
- Image generation with DALL-E 3
- Audio transcription with Whisper and text-to-speech
- Video generation basics
- Realtime Audio / Voice Agents

**Prerequisites:** Module 00 (API basics), OpenAI API key

---

## 0. Setup

In [ ]:
%pip install -q openai

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()
print("✅ Setup complete")

---
## 1. Vision — Analyze an Image from URL

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in 2 sentences."},
            {"type": "image_url", "image_url": {
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
            }}
        ]
    }],
    max_tokens=150,
)
print(response.choices[0].message.content)

---
## 2. OCR — Extract Text from Image

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Extract all visible text from this image."},
            {"type": "image_url", "image_url": {
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
            }}
        ]
    }],
    max_tokens=200,
)
print(response.choices[0].message.content)

---
## 3. Image Generation — DALL-E 3

In [ ]:
response = client.images.generate(
    model="dall-e-3",
    prompt="A minimalist diagram showing a RAG pipeline: document → embeddings → vector DB → retrieval → LLM → answer. Clean, technical style.",
    size="1024x1024",
    quality="standard",
    n=1,
)
print(f"Generated image URL: {response.data[0].url[:80]}...")
print(f"Revised prompt: {response.data[0].revised_prompt[:100]}...")

---
## 4. Audio — TTS + Whisper Transcription

In [ ]:
# Text-to-Speech
tts_response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input="Large language models have revolutionized natural language processing.",
)
audio_path = Path("test_audio.mp3")
tts_response.stream_to_file(audio_path)
print(f"✓ Audio saved to {audio_path}")

# Transcribe it back
with open(audio_path, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
    )

print(f"Transcript: {transcript.text}")
print(f"Language: {transcript.language}")
print(f"Duration: {transcript.duration:.1f}s")

# Cleanup
audio_path.unlink()

---
## 5. Multimodal Pipeline — Image → Structured Report

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": """Analyze this image and produce a JSON report:
{
  "main_subject": "...",
  "style": "...",
  "colors": ["..."],
  "suggested_caption": "..."
}""},
            {"type": "image_url", "image_url": {
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
            }}
        ]
    }],
    response_format={"type": "json_object"},
    max_tokens=300,
)

report = json.loads(response.choices[0].message.content)
for key, value in report.items():
    print(f"  {key}: {value}")

---
## When to Use Multimodal

| Scenario | Worth it? | Why |
|----------|-----------|-----|
| Extracting text from PDFs/screenshots | Yes | Vision APIs excel at OCR |
| Describing products from photos | Yes | More accurate than manual tagging |
| Analyzing charts/graphs | Yes | LLMs can read visual data |
| Generating marketing images | Maybe | Quality varies; human review needed |
| Generating marketing videos | Emerging | Improving rapidly |
| Voice-based agent interfaces | Yes | Realtime API enables low-latency voice UX |

**Multimodal is the future — every LLM app will eventually handle more than just text.**

---
## 🧪 Exercises

1. **Screenshot to Code**: Take a UI screenshot, send to GPT-4o, ask for HTML/CSS. How close is the output?

2. **Receipt Parser**: Build a function that takes a receipt photo and extracts: store, date, items, total.

3. **Multimodal Pipeline**: Build: (1) transcribe audio with Whisper, (2) summarize, (3) generate illustration with DALL-E.